# EEI6366 Lab 01: Big Data Processing with PySpark

**Dataset:** Open University Student Performance (`open_university_student_performance.csv`)  
**Size:** 150,500 rows × 9 columns  
**Source:** Provided for EEI6366 (Open University of Sri Lanka student performance across centers and courses)

This notebook fulfills **Step 2 – Prepare and Do the Lab**:
1. Study Getting Started with PySpark (completed)
2. Use a dataset with ≥ 100,000 rows
3. Explore it using **PySpark**: 4 operations
4. Explore it using **Spark SQL**: 4 operations

**Columns:** `student_id`, `course_code`, `academic_year`, `center`, `continuous_assessment_mark`, `final_exam_mark`, `overall_mark`, `grade`, `status`

## 1. Install & Initialize PySpark

In [ ]:
# Run this cell if PySpark is not already installed (Google Colab)
!pip install -q pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = (SparkSession
         .builder
         .appName("EEI6366-Lab01-Student-Performance")
         .master("local[*]")
         .getOrCreate())

spark

## 2. Load the Dataset

In [ ]:
from google.colab import files

# Upload the CSV file
print("Please upload open_university_student_performance.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = spark.read.csv(
    file_name,
    header=True,
    inferSchema=True
)

print("\nDataset:", file_name)
print("Total rows:", df.count())
print("Total columns:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)

## 3. PySpark DataFrame API – 4 Operations

### Operation 1 – Basic Exploration & Descriptive Statistics

View summary statistics of the numeric mark columns and check unique values of key categorical columns.

In [ ]:
# Summary statistics for the three mark columns
df.select(
    "continuous_assessment_mark",
    "final_exam_mark",
    "overall_mark"
).describe().show()

# How many unique courses, centers and academic years?
print("Unique courses :", df.select("course_code").distinct().count())
print("Unique centers :", df.select("center").distinct().count())
print("Academic years :")
df.select("academic_year").distinct().orderBy("academic_year").show()

### Operation 2 – Filtering and Ordering

Find students who failed the EEI6366 course and also find high performers from Colombo.

In [ ]:
# Students who failed EEI6366 - Big Data Processing (lowest marks first)
failed_bigdata = (df
    .filter(
        (F.col("course_code") == "EEI6366 - Big Data Processing") &
        (F.col("status") == "Fail")
    )
    .orderBy(F.col("overall_mark").asc())
    .select("student_id", "center", "academic_year",
            "continuous_assessment_mark", "final_exam_mark",
            "overall_mark", "grade")
)

print("Number of Failures in EEI6366:", failed_bigdata.count())
failed_bigdata.show(10, truncate=False)

# High performers in Colombo (overall_mark >= 80)
high_colombo = (df
    .filter(
        (F.col("center") == "Colombo") &
        (F.col("overall_mark") >= 80)
    )
    .orderBy(F.desc("overall_mark"))
)

print("\nHigh performers in Colombo (overall >= 80):", high_colombo.count())
high_colombo.show(5)

### Operation 3 – GroupBy and Aggregations

Calculate average marks and pass rate for each center and for each course.

In [ ]:
# Performance by Regional Center
center_stats = (df
    .groupBy("center")
    .agg(
        F.count("*").alias("total_students"),
        F.round(F.avg("overall_mark"), 2).alias("avg_overall_mark"),
        F.round(F.avg("continuous_assessment_mark"), 2).alias("avg_ca"),
        F.round(F.avg("final_exam_mark"), 2).alias("avg_exam"),
        F.round(
            F.sum(F.when(F.col("status") == "Pass", 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("pass_rate_pct")
    )
    .orderBy(F.desc("avg_overall_mark"))
)

print("Performance by Regional Center:")
center_stats.show(truncate=False)

# Performance by Course
course_stats = (df
    .groupBy("course_code")
    .agg(
        F.count("*").alias("enrollments"),
        F.round(F.avg("overall_mark"), 2).alias("avg_overall"),
        F.round(
            F.sum(F.when(F.col("status") == "Pass", 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("pass_rate_pct")
    )
    .orderBy(F.desc("avg_overall"))
)

print("\nPerformance by Course:")
course_stats.show(truncate=False)

### Operation 4 – Derived Column and Window Ranking

Create a new column that shows the gap between Continuous Assessment and Final Exam, then rank the top 3 students in each course.

In [ ]:
# New column: difference between Continuous Assessment and Final Exam
df_with_gap = df.withColumn(
    "ca_exam_gap",
    F.round(F.col("continuous_assessment_mark") - F.col("final_exam_mark"), 1)
)

# Rank students inside each course by overall_mark (highest first)
window_spec = Window.partitionBy("course_code").orderBy(F.desc("overall_mark"))

ranked = (df_with_gap
    .withColumn("rank_in_course", F.rank().over(window_spec))
    .filter(F.col("rank_in_course") <= 3)          # keep only Top 3
    .select("course_code", "student_id", "center", "overall_mark",
            "grade", "ca_exam_gap", "rank_in_course")
    .orderBy("course_code", "rank_in_course")
)

print("Top 3 students in each course:")
ranked.show(20, truncate=False)

## 4. Spark SQL – 4 Operations

First we register the DataFrame as a temporary SQL view.

In [ ]:
df.createOrReplaceTempView("student_performance")

# Quick check
spark.sql("SELECT COUNT(*) AS total_rows FROM student_performance").show()

### SQL Operation 1 – Filtering and Aggregation

Calculate number of students, average overall mark and pass rate for every combination of Academic Year and Center.

In [ ]:
spark.sql("""
SELECT
    academic_year,
    center,
    COUNT(*) AS students,
    ROUND(AVG(overall_mark), 2) AS avg_overall,
    ROUND(SUM(CASE WHEN status = 'Pass' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pass_rate_pct
FROM student_performance
GROUP BY academic_year, center
ORDER BY academic_year, avg_overall DESC
""").show(30, truncate=False)

### SQL Operation 2 – Grade Distribution with CASE

Show how many students got each grade in the EEI6366 course and the percentage of each grade.

In [ ]:
spark.sql("""
SELECT
    grade,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM student_performance
WHERE course_code = 'EEI6366 - Big Data Processing'
GROUP BY grade
ORDER BY
    CASE grade
        WHEN 'A+' THEN 1  WHEN 'A' THEN 2  WHEN 'A-' THEN 3
        WHEN 'B+' THEN 4  WHEN 'B' THEN 5  WHEN 'B-' THEN 6
        WHEN 'C+' THEN 7  WHEN 'C' THEN 8  WHEN 'C-' THEN 9
        WHEN 'D+' THEN 10 WHEN 'D' THEN 11 WHEN 'F' THEN 12
        ELSE 13
    END
""").show()

### SQL Operation 3 – Window Function

For students of EEI6366, show each student’s mark together with the average mark of their center and the difference from that average.

In [ ]:
spark.sql("""
SELECT
    student_id,
    center,
    course_code,
    overall_mark,
    ROUND(AVG(overall_mark) OVER (PARTITION BY center), 2) AS center_avg,
    ROUND(overall_mark - AVG(overall_mark) OVER (PARTITION BY center), 2) AS diff_from_center_avg
FROM student_performance
WHERE course_code = 'EEI6366 - Big Data Processing'
ORDER BY center, overall_mark DESC
LIMIT 20
""").show(truncate=False)

### SQL Operation 4 – Finding Interesting Patterns

Find students who scored high in Continuous Assessment (≥ 70) but low in the Final Exam (< 40).

In [ ]:
spark.sql("""
SELECT
    student_id,
    course_code,
    center,
    continuous_assessment_mark AS ca,
    final_exam_mark AS exam,
    overall_mark,
    grade,
    status
FROM student_performance
WHERE continuous_assessment_mark >= 70
  AND final_exam_mark < 40
ORDER BY continuous_assessment_mark DESC
LIMIT 15
""").show(truncate=False)

## 5. Key Findings

- The dataset has **150,500** records across 6 courses, 6 regional centers and 3 academic years.
- Overall pass rate is high (around 90%).
- Average marks and pass rates differ between centers.
- Some students do well in Continuous Assessment but poorly in the Final Exam.
- Window functions and group-by aggregations work well even on 150k+ rows.

## 6. Challenges

- Setting up the Spark environment (Java and SPARK_HOME).
- Choosing clear and meaningful operations that produce useful insights.

---
**Repository:** https://github.com/missarii/Big-Data-Processing-with-PySpark

In [ ]:
# Stop the Spark session
spark.stop()
print("Spark session stopped.")